# 02. Klasifikasi teks dengan embedding

Menggunakan mini-batch, representasi rata-rata embedding, baseline, macro-F1, dan analisis kesalahan.

**Prasyarat:** modul 01.

**Pola belajar:** baca penjelasan, prediksi bentuk keluaran, jalankan kode, lalu ubah satu hal.

Contoh ulasan dalam paket ini merupakan data sintetis untuk mempelajari mekanisme. Metriknya tidak mewakili kinerja pada ulasan nyata.

## Penyiapan

Instal dependensi melalui petunjuk README sebelum menjalankan seluruh sel. Setiap notebook dapat dimulai dengan kernel baru. GPU bersifat opsional. Semua operasi tensor yang berinteraksi harus berada pada perangkat yang sesuai.

In [1]:
from pathlib import Path
import sys
# Lokal: buka dari root repo, folder nlp, atau nlp/notebooks.
# Colab: ambil paket kursus jika belum tersedia.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "nlp")
             if (p / "nlp_course").is_dir()), None)
if ROOT is None and "google.colab" in sys.modules:
    import subprocess
    target = Path("/content/pytorch-deep-learning-nlp")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                        "--sparse", "--branch", "nlp-learning-path",
                        "https://github.com/FeliksMakarios/pytorch-deep-learning.git",
                        str(target)], check=True)
        subprocess.run(["git", "sparse-checkout", "set", "nlp"], cwd=target, check=True)
    ROOT = target / "nlp"
if ROOT is None or not (ROOT / "nlp_course").is_dir():
    raise RuntimeError("Folder nlp_course tidak ditemukan. Ikuti petunjuk README nlp.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch
from torch import nn
from nlp_course.data import tokenize, build_vocab, encode, read_rows, loaders, collate_batch
from nlp_course.models import MeanClassifier, RecurrentClassifier, TinyTransformer
from nlp_course.engine import seed_all, fit, run_epoch, metrics, save_mean, load_mean, predict
seed_all(42)
torch.set_num_threads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("PyTorch:", torch.__version__, "Perangkat:", device)

PyTorch: 2.14.0+cu130 Perangkat: cpu


## 1. DataLoader untuk mini-batch

DataLoader mengirim beberapa contoh sekaligus. Hanya data latih yang diacak. Panjang kalimat asli dipertahankan agar model urutan dapat memakainya kemudian.

In [2]:
vocab, train, val, test = loaders()
ids, lengths, labels = next(iter(train))
print(ids.shape, lengths.shape, labels.shape)
print(ids[:2], lengths[:2])

torch.Size([16, 6]) torch.Size([16]) torch.Size([16])
tensor([[22, 10,  4,  3,  0,  0],
        [22, 10,  8,  3,  0,  0]]) tensor([4, 4])


## 2. Menulis model sendiri

Setiap token memperoleh vektor. Mask menghapus kontribusi PAD dari rata-rata. Lapisan linear memetakan representasi kalimat ke dua kelas. Kesederhanaan ini memudahkan kita memahami apa yang hilang, yaitu urutan kata.

In [3]:
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,dim,padding_idx=0)
        self.linear = nn.Linear(dim,2)
    def forward(self, ids, lengths):
        x = self.embedding(ids)
        mask = ids.ne(0).unsqueeze(-1)
        mean = (x*mask).sum(1) / mask.sum(1).clamp_min(1)
        return self.linear(mean)
model = SentimentModel(len(vocab))
print(model(ids,lengths).shape)

torch.Size([16, 2])


## 3. Satu langkah mini-batch

Sebelum menggunakan fungsi pelatihan, periksa satu pembaruan secara langsung. Label harus berbentuk `[B]` dengan tipe long. Logits berbentuk `[B,2]`.

In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
model.train()
loss = nn.CrossEntropyLoss()(model(ids,lengths),labels)
optimizer.zero_grad()
loss.backward()
print("Norma gradien embedding:", model.embedding.weight.grad.norm().item())
optimizer.step()

Norma gradien embedding: 0.0796685516834259


## 4. Pelatihan dan validasi

Fungsi `fit` berisi siklus yang dipelajari pada modul 01. Fungsi ini menyimpan checkpoint dengan loss validasi terkecil. Baca `nlp_course/engine.py` agar abstraksi tetap dapat ditelusuri.

In [5]:
history = fit(model, train, val, epochs=20)
print(history[0])
print(history[-1])

{'epoch': 1, 'train_loss': 0.6569284995396932, 'val_loss': 0.6547797123591105, 'val_macro_f1': 0.3333333432674408}
{'epoch': 20, 'train_loss': 0.3672685805294249, 'val_loss': 0.7296118140220642, 'val_macro_f1': 0.7333333492279053}


## 5. Baseline dan macro-F1

Baseline kelas mayoritas dihitung dari label latih. Macro-F1 memberi bobot setara pada setiap kelas. Matriks kebingungan memakai baris sebagai label sebenarnya dan kolom sebagai prediksi.

In [6]:
from collections import Counter
rows = read_rows()
majority = Counter(r["label"] for r in rows if r["split"]=="train").most_common(1)[0][0]
y_test = [r["label"] for r in rows if r["split"]=="test"]
print("Baseline:", metrics(y_test,[majority]*len(y_test)))
print("Model:",run_epoch(model,test))
print("Contoh metrik manual:", metrics([0,0,1,1],[0,1,1,1]))

Baseline: {'accuracy': 0.5, 'macro_f1': 0.3333333432674408, 'confusion': [[48, 0], [48, 0]]}
Model: {'loss': 0.5947135289510092, 'accuracy': 0.625, 'macro_f1': 0.5636363625526428, 'confusion': [[12, 36], [0, 48]]}
Contoh metrik manual: {'accuracy': 0.75, 'macro_f1': 0.7333333492279053, 'confusion': [[1, 1], [0, 2]]}


## 6. Analisis kesalahan menggunakan validasi

Periksa kesalahan validasi sebelum menentukan perubahan model. Contoh negasi dapat menyingkap keterbatasan representasi rata-rata. Jangan mengubah desain berulang kali berdasarkan kesalahan data uji.

In [7]:
model.eval()
for row in [r for r in rows if r["split"]=="val"]:
    token_ids = torch.tensor([encode(row["text"],vocab)])
    with torch.inference_mode():
        pred = model(token_ids, torch.tensor([token_ids.shape[1]])).argmax(1).item()
    if pred != row["label"]:
        print("Salah:", row["text"], "benar:", row["label"], "prediksi:", pred)
        break
else:
    print("Tidak ada kesalahan pada validasi sintetis ini.")

Salah: setelah mencoba buku kesan saya buruk benar: 0 prediksi: 1


## 7. Membuktikan keterbatasan urutan

Rata-rata embedding tidak berubah jika urutan token dipermutasi. Dua kalimat berikut memiliki kumpulan token yang sama tetapi fokus negasinya berbeda. Ini merupakan alasan untuk mempelajari model urutan.

In [8]:
pair = ["baik bukan buruk", "buruk bukan baik"]
a,b = [torch.tensor([encode(t,vocab)]) for t in pair]
with torch.inference_mode():
    scores_a = model(a,torch.tensor([a.shape[1]]))
    scores_b = model(b,torch.tensor([b.shape[1]]))
print(scores_a, scores_b)
assert torch.allclose(scores_a,scores_b,atol=1e-6)

tensor([[ 0.2670, -0.0250]]) tensor([[ 0.2670, -0.0250]])


## Latihan mandiri

1. Hitung precision, recall, dan F1 kelas positif untuk contoh metrik manual.
2. Bandingkan dimensi embedding 8 dan 32 dengan seed yang sama.
3. Mengapa akurasi tinggi belum cukup?
4. Apakah softmax sama dengan keyakinan yang sudah terkalibrasi?

## Pembahasan latihan

1. TP=2, FP=1, FN=0. Precision=2/3, recall=1, F1=0.8.
2. Pilih berdasarkan validasi dan catat jumlah parameter, bukan ukuran embedding saja.
3. Ketidakseimbangan kelas, pola sintetis, dan jenis kesalahan dapat tersembunyi.
4. Tidak. Kalibrasi perlu dievaluasi terpisah.

## Penghubung ke materi berikutnya

Modul 03 mempertahankan antarmuka data yang sama dan mengganti cara model membaca urutan.

### Rujukan
- [Dokumentasi PyTorch](https://docs.pytorch.org/docs/stable/index.html)
- [Sumber Embedding](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/sparse.py)
- [Sumber Transformer](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/transformer.py)
- [Kursus sumber dan struktur awal](https://github.com/mrdbourke/pytorch-deep-learning)

Materi ini ditulis sebagai jalur NLP mandiri. Penjelasan dan contoh NLP bukan terjemahan resmi kursus sumber.